# Transcript and genomic coordinate adapter

Exons are supplied in genomic 0-based half-open coordinates. The adapter
orders them in transcript orientation, so all public spliced positions and
junction IDs are strand-safe.

In [ ]:
#| default_exp coordinates

In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable

In [ ]:
#| export
@dataclass(frozen=True)
class Exon:
    start: int
    end: int

    def __post_init__(self) -> None:
        if self.start < 0 or self.end <= self.start:
            raise ValueError("Exon must satisfy 0 <= start < end")

    @property
    def length(self) -> int:
        return self.end - self.start

In [ ]:
#| export
@dataclass(frozen=True)
class TranscriptCoordinateMap:
    transcript_id: str
    chrom: str
    strand: str
    exons: tuple[Exon, ...]

    def __post_init__(self) -> None:
        if self.strand not in {"+", "-"}:
            raise ValueError("strand must be '+' or '-'")
        if not self.exons:
            raise ValueError("At least one exon is required")
        genomic = sorted(self.exons, key=lambda exon: exon.start)
        if any(left.end > right.start for left, right in zip(genomic, genomic[1:])):
            raise ValueError("Exons must not overlap")

    @property
    def transcript_exons(self) -> tuple[Exon, ...]:
        return tuple(sorted(self.exons, key=lambda exon: exon.start, reverse=self.strand == "-"))

    @property
    def length(self) -> int:
        return sum(exon.length for exon in self.exons)

    @property
    def junction_positions(self) -> tuple[int, ...]:
        position, result = 0, []
        for exon in self.transcript_exons[:-1]:
            position += exon.length
            result.append(position)
        return tuple(result)

    def junction_id(self, spliced_position: int) -> str:
        if spliced_position not in self.junction_positions:
            raise ValueError("spliced_position is not an annotated exon junction")
        return f"{self.transcript_id}:splice:{spliced_position}"

    def spliced_to_genomic(self, spliced_position: int) -> int:
        if not 0 <= spliced_position < self.length:
            raise ValueError("spliced position outside transcript")
        remaining = spliced_position
        for exon in self.transcript_exons:
            if remaining < exon.length:
                return exon.start + remaining if self.strand == "+" else exon.end - 1 - remaining
            remaining -= exon.length
        raise AssertionError("unreachable")

    def genomic_to_spliced(self, genomic_position: int) -> int | None:
        total = 0
        for exon in self.transcript_exons:
            if exon.start <= genomic_position < exon.end:
                offset = genomic_position - exon.start if self.strand == "+" else exon.end - 1 - genomic_position
                return total + offset
            total += exon.length
        return None

In [ ]:
#| export
def coordinate_map_from_exons(transcript_id: str, chrom: str, strand: str, exons: Iterable[tuple[int, int]]) -> TranscriptCoordinateMap:
    return TranscriptCoordinateMap(transcript_id, chrom, strand, tuple(Exon(start, end) for start, end in exons))